In [26]:
import pickle   # importing pickle for saving and loading machine learning models
from sklearn.model_selection import train_test_split  # importing train_test_split for spliting the data into training and testing
from imblearn.over_sampling import SMOTE  # importing SMOTE for Balancing the Data
import warnings
warnings.filterwarnings('ignore')

In [27]:
import pandas as pd
import numpy as np
import seaborn as sns
#from ydata_profiling import ProfileReport
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder,OrdinalEncoder
import warnings
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
import pickle
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.model_selection import GridSearchCV

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

from collections import Counter

class ModifiedLabelEncoder(LabelEncoder):
    def fit_transform(self, y, *args, **kwargs):
        return super().fit_transform(y).reshape(-1, 1)

    def transform(self, y, *args, **kwargs):
        return super().transform(y).reshape(-1, 1)


In [28]:
with open('Processed_data.pkl','rb') as f:
    df = pickle.load(f)

In [29]:
with open('Tree_CT.pkl','rb') as f:
    pre = pickle.load(f)

In [30]:
x= df.drop("Attrition",axis=1)     # Extract the features (all columns except Attritions) from the dataset
y = df["Attrition"].map({"No":0,"Yes":1})  # Extract the target variable from the dataset with converting 0 and 1.

In [31]:
# =========================================================
# SPLIT DATA
# =========================================================
x_train,x_test,y_train,y_test=train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)


In [32]:
# =========================================================
# PREPROCESSING
# =========================================================

processed_x_train=pre.fit_transform(x_train)
processed_x_test=pre.transform(x_test)
smote=SMOTE(random_state=42)
x_sm,y_sm=smote.fit_resample(
    processed_x_train,
    y_train
)
print(Counter(y_sm))

Counter({0: 978, 1: 978})


In [33]:
# =========================================================
# TRAIN SCORE FUNCTION
# =========================================================

def train_score(model,x_train,y_train):

    y_pred_train=model.predict(x_train)
    train_accuracy=accuracy_score(y_train,y_pred_train)
    print("Train Accuracy :",train_accuracy)
    

def test_score(model,x_test,y_test):

    y_pred_test=model.predict(x_test)
    test_accuracy=accuracy_score(y_test,y_pred_test)
    print("Test Accuracy :",test_accuracy)
    print("\nClassification Report\n")

    print(classification_report(
        y_test,
        y_pred_test
    ))

# =========================================================

In [34]:
# =========================================================
# DECISION TREE MODEL
# =========================================================

def build_decision_tree(x_train,y_train):

    model=DecisionTreeClassifier(
        criterion='gini',
        max_depth=5,
        random_state=42
    )

    model.fit(x_train,y_train)

    return model


In [35]:

# =========================================================
# HYPERPARAMETER TUNING
# =========================================================

def tuned_decision_tree(x_train,y_train):

    params={
        'criterion':['gini','entropy'],
        'max_depth':[3,5,10]
    }

    grid_search=GridSearchCV(
        estimator=DecisionTreeClassifier(
            random_state=42
        ),
        param_grid=params,
        cv=5,
        scoring='accuracy'
    )

    grid_search.fit(x_train,y_train)

    print("Best Parameters :",
          grid_search.best_params_)

    return grid_search.best_estimator_


In [36]:
# =========================================================
# BAGGING MODEL
# =========================================================

def build_bagging_model(x_train,y_train):

    model=BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100,
        random_state=42
    )

    model.fit(x_train,y_train)

    return model



In [37]:
# =========================================================
# GRADIENT BOOSTING MODEL
# =========================================================

def build_gradient_boosting_model(x_train,y_train):

    model=GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    )

    model.fit(x_train,y_train)

    return model



In [38]:
# =========================================================
# XGBOOST MODEL
# =========================================================

def build_xgboost_model(x_train,y_train):

    model=XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    model.fit(x_train,y_train)

    return model



In [39]:
from sklearn.ensemble import RandomForestClassifier

# =========================================================
# RANDOM FOREST MODEL
# =========================================================

def build_random_forest_model(x_train,y_train):

    model=RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42
    )

    model.fit(x_train,y_train)

    return model


random_forest_model=build_random_forest_model(
    x_sm,
    y_sm
)


print("\n")
print("="*60)
print("RANDOM FOREST MODEL")
print("="*60)

train_score(
    random_forest_model,
    x_sm,
    y_sm
)

test_score(
    random_forest_model,
    processed_x_test,
    y_test
)



RANDOM FOREST MODEL
Train Accuracy : 0.9197341513292433
Test Accuracy : 0.8401360544217688

Classification Report

              precision    recall  f1-score   support

           0       0.89      0.93      0.91       255
           1       0.37      0.28      0.32        39

    accuracy                           0.84       294
   macro avg       0.63      0.60      0.61       294
weighted avg       0.82      0.84      0.83       294



# CALLING THE MODELS

In [40]:
# =========================================================
# DECISION TREE
# =========================================================

decision_tree_model=build_decision_tree(
    x_sm,
    y_sm
)

print("\nDecision Tree Train Result")

train_score(
    decision_tree_model,
    x_sm,
    y_sm
)

print("\nDecision Tree Test Result")

test_score(
    decision_tree_model,
    processed_x_test,
    y_test
)




Decision Tree Train Result
Train Accuracy : 0.8844580777096115

Decision Tree Test Result
Test Accuracy : 0.7993197278911565

Classification Report

              precision    recall  f1-score   support

           0       0.89      0.87      0.88       255
           1       0.27      0.31      0.29        39

    accuracy                           0.80       294
   macro avg       0.58      0.59      0.59       294
weighted avg       0.81      0.80      0.80       294



In [41]:
# =========================================================
# TUNED DECISION TREE
# =========================================================

tuned_dt_model=tuned_decision_tree(
    x_sm,
    y_sm
)

print("\nTuned DT Train Result")

train_score(
    tuned_dt_model,
    x_sm,
    y_sm
)

print("\nTuned DT Test Result")

test_score(
    tuned_dt_model,
    processed_x_test,
    y_test
)


Best Parameters : {'criterion': 'entropy', 'max_depth': 10}

Tuned DT Train Result
Train Accuracy : 0.9815950920245399

Tuned DT Test Result
Test Accuracy : 0.8129251700680272

Classification Report

              precision    recall  f1-score   support

           0       0.89      0.90      0.89       255
           1       0.28      0.26      0.27        39

    accuracy                           0.81       294
   macro avg       0.58      0.58      0.58       294
weighted avg       0.81      0.81      0.81       294



In [42]:

# =========================================================
# BAGGING MODEL
# =========================================================

bagging_model=build_bagging_model(
    x_sm,
    y_sm
)

print("\nBagging Train Result")

train_score(
    bagging_model,
    x_sm,
    y_sm
)

print("\nBagging Test Result")

test_score(
    bagging_model,
    processed_x_test,
    y_test
)



Bagging Train Result
Train Accuracy : 1.0

Bagging Test Result
Test Accuracy : 0.8571428571428571

Classification Report

              precision    recall  f1-score   support

           0       0.89      0.95      0.92       255
           1       0.43      0.23      0.30        39

    accuracy                           0.86       294
   macro avg       0.66      0.59      0.61       294
weighted avg       0.83      0.86      0.84       294



In [43]:
# =========================================================
# GRADIENT BOOSTING MODEL
# =========================================================

gradient_boosting_model=build_gradient_boosting_model(
    x_sm,
    y_sm
)

print("\nGradient Boosting Train Result")

train_score(
    gradient_boosting_model,
    x_sm,
    y_sm
)

print("\nGradient Boosting Test Result")

test_score(
    gradient_boosting_model,
    processed_x_test,
    y_test
)



Gradient Boosting Train Result
Train Accuracy : 0.9667689161554193

Gradient Boosting Test Result
Test Accuracy : 0.8775510204081632

Classification Report

              precision    recall  f1-score   support

           0       0.91      0.96      0.93       255
           1       0.56      0.36      0.44        39

    accuracy                           0.88       294
   macro avg       0.73      0.66      0.68       294
weighted avg       0.86      0.88      0.87       294



In [44]:

# =========================================================
# XGBOOST MODEL
# =========================================================

xgboost_model=build_xgboost_model(
    x_sm,
    y_sm
)

print("\nXGBoost Train Result")

train_score(
    xgboost_model,
    x_sm,
    y_sm
)

print("\nXGBoost Test Result")

test_score(
    xgboost_model,
    processed_x_test,
    y_test
)



XGBoost Train Result
Train Accuracy : 0.9637014314928425

XGBoost Test Result
Test Accuracy : 0.891156462585034

Classification Report

              precision    recall  f1-score   support

           0       0.91      0.97      0.94       255
           1       0.65      0.38      0.48        39

    accuracy                           0.89       294
   macro avg       0.78      0.68      0.71       294
weighted avg       0.88      0.89      0.88       294



In [45]:

# =========================================================
# FINAL MODEL COMPARISON
# =========================================================

models={
    "Decision Tree":decision_tree_model,
    "Tuned DT":tuned_dt_model,
    "Bagging":bagging_model,
    "Gradient Boosting":gradient_boosting_model,
    "XGBoost":xgboost_model
}

print("\n")
print("="*60)
print("FINAL MODEL COMPARISON")
print("="*60)

for name,model in models.items():

    train_accuracy=accuracy_score(
        y_sm,
        model.predict(x_sm)
    )

    test_accuracy=accuracy_score(
        y_test,
        model.predict(processed_x_test)
    )

    print("\n")
    print("-"*60)

    print("Model Name :",name)

    print("-"*60)

    print("Train Accuracy :",round(train_accuracy,4))

    print("Test Accuracy  :",round(test_accuracy,4))

    print("-"*60)



FINAL MODEL COMPARISON


------------------------------------------------------------
Model Name : Decision Tree
------------------------------------------------------------
Train Accuracy : 0.8845
Test Accuracy  : 0.7993
------------------------------------------------------------


------------------------------------------------------------
Model Name : Tuned DT
------------------------------------------------------------
Train Accuracy : 0.9816
Test Accuracy  : 0.8129
------------------------------------------------------------


------------------------------------------------------------
Model Name : Bagging
------------------------------------------------------------
Train Accuracy : 1.0
Test Accuracy  : 0.8571
------------------------------------------------------------


------------------------------------------------------------
Model Name : Gradient Boosting
------------------------------------------------------------
Train Accuracy : 0.9668
Test Accuracy  : 0.8776
------

| Difference IN TEST N TRAIN ACC   | Interpretation     |
| ------------- | ------------------ |
| 0% to 5%      | Very Good Model    |
| 5% to 10%     | Acceptable         |
| 10% to 15%    | Slight Overfitting |
| More than 15% | Strong Overfitting |


| Model             | Train Accuracy | Test Accuracy | Difference | Performance           |
| ----------------- | -------------- | ------------- | ---------- | --------------------- |
| Decision Tree     | 88.44%         | 79.93%        | 9%         | Acceptable            |
| Tuned DT          | 98.16%         | 81.29%        | 17%        | Overfitting           |
| Bagging           | 100%           | 85.71%        | 15%        | Slight Overfitting    |
| Random Forest     | 99%            | 86%           | 13%        | Better Generalization |
| Gradient Boosting | 96.67%         | 87.76%        | 9%         | Good Generalization   |
| XGBoost           | 96.37%         | 89.12%        | 7%         | Best Model            |


![dt](Decision_Tree.jpeg)

![Bagging Block Diagram](BAGGING.jpeg)

![boosting Block Diagram](boosting.jpeg)

![Random Block Diagram](Random_forest.jpeg)

![Hyper](hyper_paramater.jpeg)